<h2 style='color:purple' align='center'>Naive Bayes Tutorial Part 1: Predicting survival from titanic crash</h2>

## Learning objectives

By the end of this notebook, you should be able to:

1. Describe what a **Naive Bayes classifier** predicts and why it is called *naive*.
2. Explain why **GaussianNB** is used with numeric features such as `Age` and `Fare`.
3. Prepare Titanic data, train a classifier, inspect predictions, and interpret accuracy.
4. Explain and interpret `cross_val_score(GaussianNB(), X_train, y_train, cv=5)`.

**What is GaussianNB?** It is a Naive Bayes classifier that models each input feature as normally (Gaussian) distributed **within each class**. It estimates a mean and variance per feature and class, then applies Bayes' rule to compare the class probabilities. The *naive* assumption is that the features are conditionally independent given the class. In this tutorial, `Age` and `Fare` are numerical; `Pclass` and the encoded `female` indicator are discrete, so GaussianNB's distributional assumptions for those columns are approximations.

**Dataset requirement:** Keep `titanic.csv` in the same folder as this notebook before running it from the beginning.

### Why probability matters

Bayes' theorem is

$$P(C\mid X)=\frac{P(X\mid C)P(C)}{P(X)}$$

where `C` is a class (e.g., survived), `X` is the observed passenger information, `P(C)` is the prior, and `P(C|X)` is the posterior. In Naive Bayes, the individual feature likelihoods are multiplied under the conditional-independence assumption. The model chooses the class with the largest posterior probability.

**Note:** The model's `predict_proba` values are *estimated* probabilities; they are not guarantees that a specific passenger survived or did not survive.

In [1]:
# Pandas helps us load and manipulate the Titanic passenger dataset.
import pandas as pd

In [24]:
# Load the CSV file from the same folder as this notebook.
# Each row represents one passenger; 'Survived' is the label (0 = No, 1 = Yes).
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Name,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,35.0,1,0,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",3,male,35.0,0,0,373450,8.0500,NaN,S,0


## Step 1 — Select features and prepare data

We use `Pclass`, `Sex`, `Age` and `Fare` to predict the label `Survived` (`0` = did not survive, `1` = survived). We encode `Sex` into numeric form and fill missing ages so the model can use the feature matrix.

In [25]:
# Keep only the inputs used for this tutorial: Pclass, Sex, Age and Fare.
# The discarded columns are not used to train this particular model.
df.drop(['PassengerId','Name','SibSp','Parch','Ticket','Cabin','Embarked'],axis='columns',inplace=True)
df.head()

,Pclass,Sex,Age,Fare,Survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [26]:
# X (inputs) contains predictor columns.
# y (target) contains the correct survival class used for supervised learning.
inputs = df.drop('Survived',axis='columns')
target = df.Survived

In [5]:
# An alternative way of encoding Sex is shown below, but is intentionally disabled.
# The following cells instead use one-hot encoding with pd.get_dummies().
#inputs.Sex = inputs.Sex.map({'male': 1, 'female': 2})

In [27]:
# Convert Sex (a text category) into separate 0/1 indicator columns.
# For example: female passenger -> female=1, male=0.
dummies = pd.get_dummies(inputs.Sex)
dummies.head(3)

,female,male
0,0,1
1,1,0
2,1,0


In [28]:
# Join the newly created gender indicators with the other passenger inputs.
inputs = pd.concat([inputs,dummies],axis='columns')
inputs.head(3)

,Pclass,Sex,Age,Fare,female,male
0,3,male,22.0,7.2500,0,1
1,1,female,38.0,71.2833,1,0
2,3,female,26.0,7.9250,1,0


### Why keep only the `female` indicator?

With two categories, `male` and `female` contain the same information: when one is 1, the other is 0. Keeping only `female` is sufficient. This is often described in introductory tutorials as avoiding the *dummy-variable trap*; more precisely, dropping a redundant indicator is essential for some linear models with an intercept, while in GaussianNB it simply avoids duplicating the same information.

In [29]:
# Drop the original text column and one redundant binary indicator.
# With a binary category, 'female' alone distinguishes female (1) and male (0).
inputs.drop(['Sex','male'],axis='columns',inplace=True)
inputs.head(3)

,Pclass,Age,Fare,female
0,3,22.0,7.2500,0
1,1,38.0,71.2833,1
2,3,26.0,7.9250,1


In [30]:
# Identify columns with at least one missing (NaN) value.
inputs.columns[inputs.isna().any()]

Index(['Age'], dtype='object')

In [10]:
# Inspect the first 10 ages, including any missing values.
inputs.Age[:10]

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5     NaN
6    54.0
7     2.0
8    27.0
9    14.0
Name: Age, dtype: float64

In [31]:
# Replace missing ages with the mean age (simple tutorial approach).
# Best practice: in a real evaluation, fit the imputer on training data only
# (and within each CV training fold) to avoid data leakage.
inputs.Age = inputs.Age.fillna(inputs.Age.mean())
inputs.head()

,Pclass,Age,Fare,female
0,3,22.0,7.2500,0
1,1,38.0,71.2833,1
2,3,26.0,7.9250,1
3,1,35.0,53.1000,1
4,3,35.0,8.0500,0


## Step 2 — Train/test split and GaussianNB training

The **training set** teaches the model. The **test set** is reserved for an evaluation on records not used for model fitting. Here the notebook uses a 70:30 random split without a fixed random seed, so specific scores may vary on a rerun.

In [12]:
# Reserve 30% of the records as a hold-out test set.
# The remaining 70% are available for training and cross-validation.
# Without random_state, the split and resulting scores may change each run.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(inputs,target,test_size=0.3)

In [13]:
# GaussianNB models numerical features with a normal (Gaussian) distribution
# within each class. Here the classes are Survived=0 and Survived=1.
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()

In [14]:
# Estimate model parameters (class priors, means and variances) from the training set.
model.fit(X_train,y_train)

GaussianNB(priors=None, var_smoothing=1e-09)

In [15]:
# Accuracy on the held-out 30% test set = fraction correctly classified.
# Keep this test set separate from the five CV validation folds below.
model.score(X_test,y_test)

0.7835820895522388

## Step 3 — Predictions and probability estimates

`predict(...)` returns the most likely class (`0` or `1`). `predict_proba(...)` returns the model's estimated probability for each class. The two class probabilities for a given passenger add to 1.

In [16]:
# Inspect the first 10 test feature rows that will be used for prediction.
X_test[0:10]

,Pclass,Age,Fare,female,male
309,1,30.000000,56.9292,1,0
839,1,29.699118,29.7000,0,1
110,1,47.000000,52.0000,0,1
872,1,33.000000,5.0000,0,1
235,3,29.699118,7.5500,1,0
411,3,29.699118,6.8583,0,1
32,3,29.699118,7.7500,1,0
562,2,28.000000,13.5000,0,1
542,3,11.000000,31.2750,1,0
250,3,29.699118,7.2500,0,1


In [17]:
# Actual survival labels for the first 10 test passengers.
# Compare these against the predictions in the next cell.
y_test[0:10]

309    1
839    1
110    0
872    0
235    0
411    0
32     1
562    0
542    0
250    0
Name: Survived, dtype: int64

In [18]:
# Predict the class (0 = did not survive, 1 = survived) for 10 passengers.
model.predict(X_test[0:10])

array([1, 0, 0, 0, 1, 0, 1, 0, 1, 0], dtype=int64)

In [19]:
# Probability estimates for the first 10 test passengers.
# Column 0: P(Survived=0), Column 1: P(Survived=1); each row sums to 1.
model.predict_proba(X_test[:10])

array([[0.00455992, 0.99544008],
       [0.91382024, 0.08617976],
       [0.88164575, 0.11835425],
       [0.92347978, 0.07652022],
       [0.09084386, 0.90915614],
       [0.99093305, 0.00906695],
       [0.09094857, 0.90905143],
       [0.97923786, 0.02076214],
       [0.0516967 , 0.9483033 ],
       [0.9909573 , 0.0090427 ]])

## 5-fold cross-validation with GaussianNB

One train/test split produces **one** test accuracy. Cross-validation gives a broader view by repeatedly training on different portions of the **training** data.

### What does this line mean?

```python
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(), X_train, y_train, cv=5)
```

| Part | Meaning |
|---|---|
| `from sklearn.model_selection import cross_val_score` | Imports Scikit-learn's cross-validation scoring function. |
| `GaussianNB()` | Creates a new Gaussian Naive Bayes classifier. |
| `X_train` | Training input features: `Pclass`, `Age`, `Fare`, `female`. |
| `y_train` | Known labels (`Survived` = 0 or 1) for those training rows. |
| `cv=5` | Divides the **training portion** into five folds. |

**How the five iterations work:** Suppose `X_train` contains **100 rows**. For each iteration, the model trains using 80 rows and validates on 20 rows. The validation fold changes until all five folds have been used once. The actual number of rows in this Titanic notebook is different; **100 is just an illustration**.

| Run | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|:--|:--:|:--:|:--:|:--:|:--:|
| 1 | **Validate** | Train | Train | Train | Train |
| 2 | Train | **Validate** | Train | Train | Train |
| 3 | Train | Train | **Validate** | Train | Train |
| 4 | Train | Train | Train | **Validate** | Train |
| 5 | Train | Train | Train | Train | **Validate** |

**Output:** Five values, such as `[0.80, 0.85, 0.90, 0.85, 0.80]`, each representing accuracy on a different validation fold. This sample is illustrative, **not a guaranteed result** for your data.

**Important distinctions:**

- The function **re-trains** a separate GaussianNB instance in each fold; it does not re-score the already-fitted `model` from above.
- For binary classification, `cv=5` uses stratified folds by default, attempting to preserve the class proportions.
- Accuracy is the default score because no `scoring=` parameter was specified.
- The untouched `X_test` / `y_test` from the earlier 70:30 split are **not** used by this function.
- This notebook fills missing ages before splitting, which is fine for illustrating syntax but risks leakage. For rigorous cross-validation, put imputation in a `Pipeline` so it is fitted separately inside each training fold.

In [34]:
# cross_val_score creates and fits a NEW GaussianNB model in each fold.
# It uses only X_train and y_train here, NOT the untouched hold-out test set.
# cv=5 means five folds: four folds train, one fold validates, repeated 5 times.
# For a classifier with categorical class labels, the default is stratified folds.
# The result is an array of five validation accuracies (default scoring).
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(),X_train, y_train, cv=5)

array([0.75396825, 0.784     , 0.76612903, 0.82258065, 0.77419355])

### Interpreting all five scores

The five returned scores can be summarized with a **mean cross-validation accuracy** and **standard deviation**. The mean tells you the typical validation accuracy; the standard deviation shows how much the accuracy changes across folds. These are **validation** scores, not independent final test scores.

In [ ]:
# Run five separate train/validation evaluations on the training portion.
cv_scores = cross_val_score(GaussianNB(), X_train, y_train, cv=5)

# Scores from each validation fold (accuracy by default).
print("Accuracy in each fold:", cv_scores)

# Average validation accuracy across all five folds.
print("Mean CV accuracy:", cv_scores.mean())

# Spread of scores: smaller variation suggests more consistent folds.
print("Standard deviation:", cv_scores.std())

# Convert the average from a decimal into a more readable percentage.
print(f"Mean CV accuracy (%): {cv_scores.mean() * 100:.2f}%")

### Cross-validation versus test accuracy

- `model.score(X_test, y_test)` evaluates **one fitted model** on the original hold-out test set.
- `cross_val_score(GaussianNB(), X_train, y_train, cv=5)` trains **five separate models** on overlapping training folds and reports **five validation accuracies**.

**Exercise for students:** Explain why the 5-fold mean may differ from the score on `X_test`. Then try changing `cv=5` to `cv=3` or `cv=10` (if class sizes allow) and compare the results. Keep the independent test set for final evaluation.